# Transfer learning MobileNetV3-Small (BISINDO A-Z)

Notebook tipis: semua logika ada di `src/train_transfer.py`. Notebook ini hanya mengatur eksperimen dan menampilkan hasil.

- **Data dan split:** dari `configs/config.yaml` (`data_dir`, `split_path` = split beku). Notebook ini tidak membuat atau mengubah split.
- **Hyperparameter dasar:** `configs/transfer.yaml`; ubah yang perlu di sel konfigurasi.
- **Satu run = satu perubahan** dari run pembanding. Hasil di `results/transfer/<run_name>/`.
- **Tuning pakai val.** Set `EVALUATE_TEST = False` selama tuning; nyalakan sekali untuk hasil akhir.
- Training di CPU bisa lama. Epoch pertama mencetak waktunya; hentikan dengan tombol Interrupt bila perlu, model terbaik sejauh itu tetap disimpan dan dievaluasi.

In [5]:
import sys, json
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "configs").exists():      # dijalankan dari notebooks/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Image as IPImage, display
import train_transfer as tt
print("root repo:", ROOT)

root repo: c:\Sem5_Projects\Bisindo-cv


## 1. Konfigurasi eksperimen

In [6]:
# None = pakai nilai dari configs/transfer.yaml
OVERRIDES = dict(
    run_name="mnv3s_head2",     # nama unik per eksperimen (run yang sudah ada tidak ditimpa)
    mode="head",               # head = baseline | partial = metode utama | full
    unfreeze_blocks=None,      # hanya mode partial, mis. 2 atau 3
    epochs=None,
    pretrained=None,
    # lr_head=None, lr_backbone=None, augment=None, class_weights=None, seed=None, overwrite=None
)
EVALUATE_TEST = True           # False selama tuning

cfg = tt.build_cfg(OVERRIDES)
show = {k: v for k, v in cfg.items() if k not in ("data_dir", "split_path", "label_map_path", "results_dir")}
print(json.dumps(show, indent=2))
print("split:", tt._rel(cfg["split_path"]))

{
  "run_name": "mnv3s_head2",
  "mode": "head",
  "unfreeze_blocks": 2,
  "pretrained": true,
  "batch_size": 32,
  "epochs": 20,
  "lr_head": 0.001,
  "lr_backbone": 0.0001,
  "weight_decay": 0.0001,
  "class_weights": false,
  "augment": true,
  "patience": 6,
  "num_workers": 0,
  "draft_size": 448,
  "device": "auto",
  "overwrite": false,
  "seed": 42,
  "img_size": 224
}
split: splits/split_v2.csv


## 2. Cek data dan split

In [7]:
loaders, df = tt.make_loaders(cfg)          # juga memeriksa file contoh ada di data_dir
print(df["split"].value_counts().to_dict())
display(pd.crosstab(df["label"], df["split"])[["train", "val", "test"]].T)
x, y = next(iter(loaders["train"]))
print("satu batch:", tuple(x.shape), "| label", int(y.min()), "-", int(y.max()))

{'train': 3114, 'val': 675, 'test': 673}


label,A,B,C,D,E,F,G,H,I,J,...,Q,R,S,T,U,V,W,X,Y,Z
split,,,,,,,,,,,,,,,,,,,,,
train,115,117,117,117,117,121,121,121,121,117,...,121,119,121,121,121,120,118,122,122,122
val,25,26,26,26,26,26,26,26,26,26,...,26,26,26,26,26,26,26,26,26,26
test,25,25,26,26,25,26,26,26,26,26,...,26,26,26,26,26,26,26,26,26,26


satu batch: (32, 3, 224, 224) | label 0 - 25


## 3. Latih

In [ ]:
result = tt.run(cfg, evaluate_test=EVALUATE_TEST)

Run: mnv3s_head2 | mode=head | device=cpu | torch 2.15.0+cpu
Split: splits/split_v2.csv (md5 f71002d1) | {'train': 3114, 'val': 675, 'test': 673}
Parameter: total 1,544,506 | dilatih 26,650 (1.73%)
Epoch 01/20 | train loss 2.8120 acc 0.217 | val loss 2.4573 acc 0.308 f1 0.282 | 142s
Epoch 02/20 | train loss 2.1384 acc 0.415 | val loss 2.1484 acc 0.364 f1 0.357 | 154s
Epoch 03/20 | train loss 1.8315 acc 0.509 | val loss 2.0082 acc 0.391 f1 0.386 | 131s
Epoch 04/20 | train loss 1.6532 acc 0.558 | val loss 1.9114 acc 0.396 f1 0.386 | 128s


## 4. Hasil run ini

In [ ]:
out, m = result["out_dir"], result["metrics"]
split = m["report_split"]
print(f"model terbaik: epoch {m['best_epoch']} | val macro-F1 {m['best_val_macro_f1']:.3f}")
print(f"[{split}]", {k: round(v, 4) for k, v in (m["test"] or m["val"]).items()})
print("latency (ms):", {k: round(v, 1) for k, v in m["latency_ms"].items() if k != "n"})
if "catatan" in m:
    print("CATATAN:", m["catatan"])
display(IPImage(str(out / "curves.png")))

In [ ]:
display(IPImage(str(out / f"confusion_matrix_{split}.png")))
pc = pd.read_csv(out / f"per_class_{split}.csv").sort_values("f1")
print("Kelas paling lemah:")
display(pc.head(8))
print("Pasangan huruf yang paling sering tertukar:")
display(pd.read_csv(out / f"top_confusions_{split}.csv").head(10))
display(IPImage(str(out / f"sample_predictions_{split}.png")))

## 5. Perbandingan semua run

In [ ]:
cmp = tt.compare_runs()                      # juga menyimpan results/transfer/comparison.csv
cmp

## Catatan
- **Eksperimen terkontrol:** ubah satu hal per run (mis. `mode` head -> partial, atau `unfreeze_blocks` 2 -> 3, atau `augment`), nama run baru, split dan seed sama. Tabel di atas menunjukkan perbedaannya.
- **Keterbatasan split:** `split_v2` tidak memakai grup perangkat/latar, sehingga kondisi foto yang sama bisa ada di train dan test dan skor bisa lebih optimis daripada kondisi nyata. Jangan membandingkan angka dengan notebook lintas-kondisi (dataset Kaggle).
- **Ketidakseimbangan per kelas** kecil, tetapi laporkan macro-F1 dan akurasi per kelas, bukan hanya accuracy.
- **Demo:** `python src/demo.py --run <run_name> --image foto.jpg` (offline) atau `--webcam` (jalankan di terminal, bukan di notebook).